# 🧪 Assay Statistics Summary (Notebook 04)

This notebook summarizes the filtered PubChem assays:
- Total assays per pathogen
- Number and % of assays with ChEMBL ID
- Global % with ChEMBL ID

In [ ]:
import pandas as pd
from pathlib import Path

In [ ]:
# Set paths
DATA_PROCESSED = Path("../data/processed")

## 1. Load filtered assay metadata (v2)

In [ ]:
df = pd.read_csv(DATA_PROCESSED / "filtered_description_with_organisms_v2_REBUILT.csv")
df["Pathogen_list"] = df["Pathogen"].str.split(", ")

# Explode: one row per (AID, Pathogen)
df_exp = df.explode("Pathogen_list", ignore_index=True)
df_exp = df_exp.rename(columns={"Pathogen_list": "Pathogen"})
df_exp = df_exp.dropna(subset=["Pathogen"])

## 2. Count total assays per pathogen

In [ ]:
summary_total = (
    df_exp.groupby("Pathogen")["AID"]
    .nunique()
    .reset_index(name="Total_AIDs")
)

## 3. Count assays with ChEMBL ID

In [ ]:
df_exp["Has_ChEMBL"] = df_exp["ChEMBLid"].notna()

summary_chembl = (
    df_exp[df_exp["Has_ChEMBL"]]
    .groupby("Pathogen")["AID"]
    .nunique()
    .reset_index(name="AIDs_with_ChEMBL")
)

## 4. Merge and compute percentages

In [ ]:
df_summary = summary_total.merge(summary_chembl, on="Pathogen", how="left")
df_summary["AIDs_with_ChEMBL"] = df_summary["AIDs_with_ChEMBL"].fillna(0).astype(int)
df_summary["Percent_with_ChEMBL"] = (
    100 * df_summary["AIDs_with_ChEMBL"] / df_summary["Total_AIDs"]
).round(1)

## 5. Sort and display

In [ ]:
df_summary = df_summary.sort_values("Total_AIDs", ascending=False)
df_summary

## 6. Global statistics

In [ ]:
total_assays = df_summary["Total_AIDs"].sum()
total_with_chembl = df_summary["AIDs_with_ChEMBL"].sum()
global_percentage = 100 * total_with_chembl / total_assays

print(f"🌍 Global summary:")
print(f"- Total filtered assays: {total_assays:,}")
print(f"- With ChEMBL ID: {total_with_chembl:,} ({global_percentage:.1f}%)")

## ✅ Save summary to file

In [ ]:
df_summary.to_csv(DATA_PROCESSED / "summary_with_chembl_per_pathogen.csv", index=False)